---   
 <img align="left" width="75" height="75"  src="https://upload.wikimedia.org/wikipedia/en/c/c8/University_of_the_Punjab_logo.png"> 

<h1 align="center">Department of Data Science</h1>

---
<h3><div align="right">Instructor: Muhammad Arif Butt, Ph.D.</div></h3>    

<br><br>
<h1 align="center">Lec-21: Loading and Running OS LLMs using AutoModel/AutoTokenizer</h1>

# Learning agenda of this notebook
1. Recap and Ways to Access Open Source LLMs
2. Downloading Models and Peeping Inside Their Components
3. Download TinyLlama Model, its Configuration and Tokenizer
4. Inference Techniques: an Overview
5. Inference using Transformers Library **`pipeline()`** Method
6. Inference using Model's **`generate()`** Method
7. Inference using a **Manual Forward Pass** (and Speeding it up with the **KV Cache**)

# <span style='background :lightgreen' >1. Recap and Ways to Access Open Source LLMs</span>

<div style="text-align:center;">
    <img src="../images/tfmr-007.png"
         style="max-width:1500px; width:100%; height:auto; display:inline-block;">
</div>


## Ways to Access Open Source LLMs
### (i) Access Open-Source Models via Cloud-Based Providers
### (ii) Run Open-Source Models locally using runtimes
### (iii) Use Open-Source Models via Transformers `pipeline()` API
### (iv) Load models using Transformers `AutoModel/AutoTokenizer` Classes
### (v) Fine-Tune LLMs using full fine-tuning or PEFT methods (LoRA / QLoRA / adapters) 
### (vi) Build and train an AI Model from scratch using PyTorch / TensorFlow 

# <span style='background :lightgreen' >2. Downloading Models and Peeping Inside Their Components</span>

- **Option 1:** You can use architecture-specific loaders like:
    - `LlamaForCausalLM.from_pretrained()`         → LLaMA 1, LLaMA 2, LLaMA 3, TinyLlama, Vicuna, Alpaca
    - `MistralForCausalLM.from_pretrained()`       → Mistral 7B, Mistral variants
    - `MixtralForCausalLM.from_pretrained()`       → Mixtral 8x7B (Mixture of Experts)
    - `GPT2LMHeadModel.from_pretrained()`          → GPT-2 all variants
    - `GemmaForCausalLM.from_pretrained()`         → Gemma (Google); Gemma 2 uses `Gemma2ForCausalLM`
    - `Phi3ForCausalLM.from_pretrained()`          → Phi-3 (Microsoft)
    - `Qwen2ForCausalLM.from_pretrained()`         → Qwen2, Qwen2.5 (Alibaba); Qwen3 uses `Qwen3ForCausalLM`
- **Option 2:** You can use the following AutoModel classes, which automatically detect the model architecture from the `config.json` file in the model repository:
    - `AutoModelForCausalLM.from_pretrained()`   → decoder-only models: GPT-2, LLaMA-3, Mistral, Qwen, TinyLlama
    - `AutoModel.from_pretrained()`              → BERT-Base, BERT-Large, DistilBERT (returns the bare model without a task head)
    - `AutoModelForSeq2SeqLM.from_pretrained()`  → BART-Base, BART-Large, T5-Base, T5-Large
- The above loaders will download the model files (weights, config) from the Hugging Face Hub (first time only)
- Stores them in your local HF cache (`~/.cache/huggingface/`) (first time only)
- Loads all model layers from cache into local RAM / GPU memory
- Initializes the architecture (transformer blocks, embeddings, attention layers)
- Applies any optimizations you specify (e.g., 4-bit quantization, device_map="auto")
- The returned `model` object is a neural network, based on the Transformer architecture, implemented with the Python framework PyTorch.
>- **Note on `trust_remote_code`:** we do **not** pass `trust_remote_code=True` in this notebook, because all the models used here are natively supported by the transformers library. That flag allows the model repository to run its own Python code on your machine, so use it only for repositories that you trust and that really need it.

In [1]:
#################################        DECODER-ONLY Models        #####################################
# GPT-1-117M       │  12 layers  │  d_model=768    │  vocab=40478  (BPE) │   512 tokens  
# GPT-3-175B       │  96 layers  │  d_model=12288  │  vocab=50257  (BPE) │  2048 tokens  
# Llama-3.2-1B     │  16 layers  │  d_model=2048   │  vocab=128256 (BPE) │  128K tokens  
# Llama-3.1-8B     │  32 layers  │  d_model=4096   │  vocab=128256 (BPE) │  128K tokens
########################################################################################################
# NOTE: meta-llama/Llama-3.2-1B is a GATED model. Before running this cell, open its page on Hugging Face,
#       accept Meta's license, wait for approval, and log in (e.g., `hf auth login` or set HF_TOKEN).
#       Otherwise you will get a 401/403 error. You can also try an ungated model such as "Qwen/Qwen2.5-0.5B".
from transformers import AutoModelForCausalLM, LlamaForCausalLM
from transformers import AutoTokenizer
import rich

model = AutoModelForCausalLM.from_pretrained(
                                            "meta-llama/Llama-3.2-1B",
                                            device_map="cpu",
                                            dtype="auto",        # Automatically chooses the best dtype. Can specify torch.float16, torch.float32 (fp32, fp16, int8, etc.)
)

memory = model.get_memory_footprint() / 1e6 
print(f"Memory Footprint: {memory:,.1f} MB")
rich.print(model)

Memory Footprint: 2,471.6 MB


LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 2048)
    (layers): ModuleList(
      (0-15): 16 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=512, bias=False)
          (v_proj): Linear(in_features=2048, out_features=512, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((2048,), eps=1e-05)
    (rotary_emb): LlamaRotaryEmbedding()
  )
  (lm_head): Linear(in_features=2048, out_features=128256, bias=False)
)

- **Connecting the printed model to Lec-19 and Lec-20.** Llama-3.2-1B uses several of the later improvements that we studied:

| Printed module | What it is | Covered in |
|---|---|---|
| `embed_tokens: Embedding(128256, 2048)` | Token embedding matrix (vocab_size × d_model) | Lec-19 |
| `rotary_emb: LlamaRotaryEmbedding` | **RoPE**: there is no position-embedding table; Q and K are rotated inside every attention layer | Lec-19 §7 |
| `16 x LlamaDecoderLayer` | A decoder-only stack (no encoder, so **no cross-attention**) | Lec-19 §6 |
| `q_proj: 2048 → 2048`, `k_proj` / `v_proj: 2048 → 512` | 32 query heads × 64, but only 8 KV heads × 64: **Grouped-Query Attention**, each K, V head is shared by 4 query heads (a 4× smaller KV cache) | Lec-20 §7, §10 |
| `o_proj: 2048 → 2048` | The output matrix W^O of multi-head attention | Lec-20 §7 |
| `gate_proj`, `up_proj: 2048 → 8192`, `down_proj: 8192 → 2048`, `SiLU` | A **SwiGLU** feed-forward network with d_ff = 8192: `down(SiLU(gate(x)) * up(x))` | Lec-20 §8 |
| `input_layernorm`, `post_attention_layernorm: LlamaRMSNorm` | **RMSNorm**, applied *before* attention and *before* the MLP (**Pre-LN**) | Lec-19 §6, Lec-20 §8 |
| `norm: LlamaRMSNorm` | The final norm before the LM head | Lec-20 §9 |
| `lm_head: 2048 → 128256` | The LM head; in Llama-3.2-1B it **shares (ties)** its weights with `embed_tokens` | Lec-20 §9 |

- Note also `bias=False` in every Linear layer: most modern LLMs drop the bias terms.

In [2]:
#################################        ENCODER-ONLY Models        #####################################
# BERT-Base-110M     │ 12 layers │ d_model=768  │ vocab=30522 (Word Piece)│ 512 tokens  
# BERT-Large-340M    │ 24 layers │ d_model=1024 │ vocab=30522 (Word Piece)│ 512 tokens
# DistilBERT-66M     │  6 layers │ d_model=768  │ vocab=30522 (Word Piece)│ 512 tokens
# RoBERTa-Base-125M  │ 12 layers │ d_model=768  │ vocab=50265 (BPE)       │ 512 tokens  
# RoBERTa-Large-355M │ 24 layers │ d_model=1024 │ vocab=50265 (BPE)       │ 512 tokens  
########################################################################################################

from transformers import AutoModel
import rich

# download BERT-Base — a standard encoder-only model
# Note: the load report below lists some 'UNEXPECTED' cls.* weights. They belong to BERT's pre-training heads,
#       which AutoModel drops because it returns the bare model without a task head. They can be ignored.
model     = AutoModel.from_pretrained(
                        "bert-base-uncased",
                        device_map        = "cpu",
                        dtype       = "auto",        # Automatically chooses the best dtype. Can specify torch.float16, torch.float32 (fp32, fp16, int8, etc.)
                        )

memory = model.get_memory_footprint() / 1e6
print(f"Memory Footprint: {memory:,.1f} MB")
rich.print(model)

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Memory Footprint: 437.9 MB


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
        )
        (intermediate): BertIntermediate(
          (dense): Linear(in_features=768, out_features=3072, bias=True)
          (intermediate_act_fn): GELUActivation()
        )
        (output): BertOutput(
          (dense): Linear(in_features=3072, out_features=768, bias=True)
          (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
  )
  (pooler): BertPooler(
    (dense): Linear(in_features=768, out_features=768, bias=True)
    (activation): Tanh()
  )
)

In [3]:
#################################        ENCODER-DECODER Models        #####################################
# BART-Base-139M   │  6 enc +  6 dec  │  d_model=768   │  vocab=50265 (BPE)           │  1024 tokens  
# BART-Large-406M  │ 12 enc + 12 dec  │  d_model=1024  │  vocab=50265 (BPE)           │  1024 tokens  
# T5-Base-220M     │ 12 enc + 12 dec  │  d_model=768   │  vocab=32128 (Sentence Piece)│   512 tokens  
# T5-Large-770M    │ 24 enc + 24 dec  │  d_model=1024  │  vocab=32128 (Sentence Piece)│   512 tokens
###########################################################################################################

from transformers import AutoModelForSeq2SeqLM
import rich

model = AutoModelForSeq2SeqLM.from_pretrained(
                                    "facebook/bart-base",
                                    device_map = "cpu",
                                    dtype  = "auto",        # Automatically chooses the best dtype. Can specify torch.float16, torch.float32 (fp32, fp16, int8, etc.)
                                    )

memory = model.get_memory_footprint() / 1e6
print(f"Memory Footprint: {memory:,.1f} MB")
rich.print(model)

Memory Footprint: 557.9 MB


BartForConditionalGeneration(
  (model): BartModel(
    (shared): BartScaledWordEmbedding(50265, 768, padding_idx=1)
    (encoder): BartEncoder(
      (embed_tokens): BartScaledWordEmbedding(50265, 768, padding_idx=1)
      (embed_positions): BartLearnedPositionalEmbedding(1026, 768)
      (layers): ModuleList(
        (0-5): 6 x BartEncoderLayer(
          (self_attn): BartAttention(
            (k_proj): Linear(in_features=768, out_features=768, bias=True)
            (v_proj): Linear(in_features=768, out_features=768, bias=True)
            (q_proj): Linear(in_features=768, out_features=768, bias=True)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=768, out_features=3072, bias=True)
          (fc2): Linear(in_features=3072, out_features=768, bias=True)
          (final_layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        )
      )
      (layernorm_embedding): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    )
    (decoder): BartDecoder(
      (embed_tokens): BartScaledWordEmbedding(50265, 768, padding_idx=1)
      (embed_positions): BartLearnedPositionalEmbedding(1026, 768)
      (layers): ModuleList(
        (0-5): 6 x BartDecoderLayer(
          (self_attn): BartAttention(
            (k_proj): Linear(in_features=768, out_features=768, bias=True)
            (v_proj): Linear(in_features=768, out_features=768, bias=True)
            (q_proj): Linear(in_features=768, out_features=768, bias=True)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (activation_fn): GELUActivation()
          (self_attn_layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (encoder_attn): BartAttention(
            (k_proj): Linear(in_features=768, out_features=768, bias=True)
            (v_proj): Linear(in_features=768, out_features=768, bias=True)
            (q_proj): Linear(in_features=768, out_features=768, bias=True)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (encoder_attn_layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (fc1): Linear(in_features=768, out_features=3072, bias=True)
          (fc2): Linear(in_features=3072, out_features=768, bias=True)
          (final_layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        )
      )
      (layernorm_embedding): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    )
  )
  (lm_head): Linear(in_features=768, out_features=50265, bias=False)
)

# <span style='background :lightgreen' >3. Download TinyLlama Model, its Configuration and Tokenizer</span>

## a. Download Model and Understand its Architecture

In [4]:
from transformers import AutoModelForCausalLM
import rich

model = AutoModelForCausalLM.from_pretrained(
                                            "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
                                            device_map="cpu",
                                            dtype="auto",        # Automatically chooses the best dtype. Can specify torch.float16, torch.float32 (fp32, fp16, int8, etc.)
)

memory = model.get_memory_footprint() / 1e6 
print(f"Memory Footprint: {memory:,.1f} MB")
rich.print(model)

Memory Footprint: 2,200.1 MB


LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 2048)
    (layers): ModuleList(
      (0-21): 22 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=256, bias=False)
          (v_proj): Linear(in_features=2048, out_features=256, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=5632, bias=False)
          (up_proj): Linear(in_features=2048, out_features=5632, bias=False)
          (down_proj): Linear(in_features=5632, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((2048,), eps=1e-05)
    (rotary_emb): LlamaRotaryEmbedding()
  )
  (lm_head): Linear(in_features=2048, out_features=32000, bias=False)
)

## b. Download the Model Config File using `AutoConfig` Class and Peep inside its Components
- Every model has an associated config file (JSON format) that stores all architectural and hyperparameter settings required to build the model structure before loading the weights. Hugging Face automatically loads it when you call `.from_pretrained()` method.
    - If the model is NOT already downloaded, the `config.json` is downloaded from Hugging Face Hub.
    - If the model is already downloaded, the `config.json` is loaded from the local cache or the folder you specify.


In [5]:
from transformers import AutoConfig

config = AutoConfig.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0")
rich.print(config)

LlamaConfig {
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 1,
  "dtype": "bfloat16",
  "eos_token_id": 2,
  "head_dim": 64,
  "hidden_act": "silu",
  "hidden_size": 2048,
  "initializer_range": 0.02,
  "intermediate_size": 5632,
  "max_position_embeddings": 2048,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 32,
  "num_hidden_layers": 22,
  "num_key_value_heads": 4,
  "pad_token_id": null,
  "pretraining_tp": 1,
  "rms_norm_eps": 1e-05,
  "rope_parameters": {
    "rope_theta": 10000.0,
    "rope_type": "default"
  },
  "tie_word_embeddings": false,
  "transformers_version": "5.7.0",
  "use_cache": true,
  "vocab_size": 32000
}

## c. Download the Model Tokenizer using `AutoTokenizer` Class and Peep inside its Components
- The `AutoTokenizer.from_pretrained()` method will:
    - Downloads the tokenizer files and loads them locally into memory.
    - Tokenizer is downloaded and stored in the cache  on first run, so that may take a bit of time. Subsequent runs will be quick
- The TinyLlama uses **LlamaTokenizerFast**, which is a highly optimized tokenizer that is specifically designed for **LLaMA-style vocabulary + BPE encoding**. This ensures that input text is **tokenized exactly the same way the model was trained**, making it essential for correct inference. This tokenizer:
    - Uses **LLaMA vocabulary + SentencePiece/BPE**
    - Converts text to integers efficiently
    - Applies BOS, EOS, UNK, PAD correctly
    - Works at high speed (FastTokenizer)
    - Matches TinyLlama’s architecture and expected training format
    - Knows the model's context length (`model_max_length` = 2048 tokens), but it truncates longer inputs only if you pass `truncation=True`

In [6]:
# Load tokenizer
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0")
rich.print(tokenizer)

LlamaTokenizer(name_or_path='TinyLlama/TinyLlama-1.1B-Chat-v1.0', vocab_size=32000, model_max_length=2048, 
padding_side='right', truncation_side='right', special_tokens={'bos_token': '<s>', 'eos_token': '</s>', 
'unk_token': '<unk>', 'pad_token': '</s>'}, added_tokens_decoder={
        0: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
        1: AddedToken("<s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
        2: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
})

In [7]:
# Find where the TinyLlama files are stored in the local Hugging Face cache (works on any operating system)
import os
from huggingface_hub import snapshot_download

local_path = snapshot_download("TinyLlama/TinyLlama-1.1B-Chat-v1.0", local_files_only=True)   # no download: just returns the local folder
print(local_path)
for file_name in sorted(os.listdir(local_path)):
    print("   ", file_name)

/Users/arif/.cache/huggingface/hub/models--TinyLlama--TinyLlama-1.1B-Chat-v1.0/snapshots/fe8a4ea1ffedaf415f4da2f062534de366a451e6
    config.json
    generation_config.json
    model.safetensors
    special_tokens_map.json
    tokenizer.json
    tokenizer.model
    tokenizer_config.json


## d. Encode and Decode the Text using Tokenizer

In [8]:
from transformers import AutoTokenizer

# Load the tokenizer, that will convert raw text into token IDs the model can understand and process
tokenizer = AutoTokenizer.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0")

prompt = "Who is Ada Lovelace?"
encoded_prompt = tokenizer.encode(prompt)

# print the prompt
print(f"prompt: {prompt}")
print(f"encoded: {encoded_prompt}")

# print out as a loop
for item in encoded_prompt:
  decoded = tokenizer.decode(item)
  print(f"|{item}|:|{decoded}|")

# print out the length
print()
print(f"length: {len(encoded_prompt)}")

prompt: Who is Ada Lovelace?
encoded: [1, 11644, 338, 23255, 23974, 295, 815, 29973]
|1|:|<s>|
|11644|:|Who|
|338|:|is|
|23255|:|Ada|
|23974|:|Lov|
|295|:|el|
|815|:|ace|
|29973|:|?|

length: 8


## e. The `apply_chat_template()` Method
- The `apply_chat_template()` method converts a structured conversation (messages with roles) into a single prompt string that the model can understand. With `tokenize=True` it returns the token IDs of that prompt, ready to be fed to the model’s `generate()` method.
- TinyLlama (and other chat-optimized LLMs) are instruction/chat models. They expect messages in a structured format
- When you use `generate()` or a manual forward pass, you must call `apply_chat_template()` yourself (§6 and §7). A `text-generation` pipeline can do it for you: if you pass it the list of messages, it applies the model's chat template automatically (§5).

In [9]:
from transformers import AutoTokenizer

# Load the tokenizer, that will convert raw text into token IDs the model can understand and process
tokenizer = AutoTokenizer.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0")

# Define a sample conversation as a list of messages structured with roles to help the chat model understand context
messages = [{"role": "system", "content": "You are a helpful assistant."}, {"role": "user", "content": "Who is Ada Lovelace?"}]

# Convert chat messages into token IDs for the model using apply_chat_template() method of the tokenizer
input_ids = tokenizer.apply_chat_template(
                                        messages,
                                        tokenize=True,              # tokenize=True converts text to token IDs
                                        return_tensors="pt",        # return_tensors="pt" returns PyTorch tensors
                                        return_dict=False,          # transformers v5: without this it returns a BatchEncoding, not a tensor
                                        add_generation_prompt=True  # Adds special tokens for model generation
)

# Inspect token IDs These are the numerical representation of each token in the conversation
print("Token IDs:\n", input_ids)

# Decode the token IDs back to text.
decoded_text = tokenizer.decode(input_ids[0], skip_special_tokens=False)
print("\nDecoded text:\n", decoded_text)

Token IDs:
 tensor([[  529, 29989,  5205, 29989, 29958,    13,  3492,   526,   263,  8444,
         20255, 29889,     2,    13, 29966, 29989,  1792, 29989, 29958,    13,
         22110,   338, 23255, 23974,   295,   815, 29973,     2,    13, 29966,
         29989,   465, 22137, 29989, 29958,    13]])

Decoded text:
 <|system|>
You are a helpful assistant.</s>
<|user|>
Who is Ada Lovelace?</s>
<|assistant|>



# <span style='background :lightgreen' >4. Inference Techniques: an Overview</span>
| # | Method | Runs On | Download Required | Complexity | Speed | Control Level | Best For |
|---|---|---|---|---|---|---|---|
| 1 | **Inference API** | HF Servers | ❌ No | ⭐ Easiest | 🚀 Fast | 🔅 Low | Quick testing, demos |
| 2 | **Direct Pipeline** | Local CPU/GPU | ✅ Yes (~2.2GB) | ⭐⭐ Easy | 🚗 Medium | 🔆 Medium | Simple local inference |
| 3 | **AutoModel + Pipeline** | Local CPU/GPU | ✅ Yes (~2.2GB) | ⭐⭐⭐ Moderate | 🚗 Medium | 🔆🔆 High | Controlled local inference |
| 4 | **AutoModel + Generate** | Local CPU/GPU | ✅ Yes (~2.2GB) | ⭐⭐⭐⭐ Advanced | 🚗 Medium | 🔆🔆🔆 Very High | Custom Python applications |
| 5 | **Manual Forward Pass** | Local CPU/GPU | ✅ Yes (~2.2GB) | ⭐⭐⭐⭐⭐ Expert | 🐢 Slow | 🔆🔆🔆🔆 Maximum | Research & learning |

- Methods 1 and 2 were covered in earlier lectures (Lec-07 and Lec-12). This notebook covers methods 3, 4 and 5.

### Method Descriptions
- **Method 1 — Hugging Face Inference API:**
    - Sends your prompt to Hugging Face servers via a simple HTTP request — **no model download, no local GPU needed**
    - Requires only a free Hugging Face account and an API token
    - Best for: quick testing, sharing demos, prototyping without any local setup
    - Limitation: rate-limited on free tier, requires internet connection, low control over generation parameters
- **Method 2 — Direct Pipeline:**
    - Uses Hugging Face `pipeline()` — the highest-level abstraction for running models locally
    - Handles tokenization, inference, and decoding automatically in just **2–3 lines of code**
    - Best for: beginners, simple local inference, when you do not need fine-grained control
    - Limitation: limited control over generation parameters and tokenization details
- **Method 3 — AutoModel + Pipeline:**
    - Loads the model and tokenizer separately using `AutoModelForCausalLM` and `AutoTokenizer`, then passes them into `pipeline()`
    - Gives you control over **how the model is loaded** (precision, device, quantization) while keeping the simple pipeline interface for generation
    - Best for: when you need custom loading options (e.g., `dtype=torch.float16`) but still want a clean generation interface
- **Method 4 — AutoModel + Generate:**
    - Loads model and tokenizer separately, manually tokenizes the prompt, and calls `model.generate()` directly
    - Gives **full control** over every generation parameter: `temperature`, `top_p`, `top_k`, `repetition_penalty`, `max_new_tokens`, beam search, sampling strategies etc.
    - Best for: building your own Python applications on top of the model, with fine-grained output control
    - This is the method most often used when you write your own Python code around a model. (For serving many users in production, dedicated inference engines such as vLLM, TGI or Ollama are normally used.)
- **Method 5 — Manual Forward Pass:**
    - Bypasses `generate()` entirely — manually runs the forward pass through the model, extracts raw logits, applies softmax, and samples the next token one at a time in a loop
    - Gives **complete visibility** into every step of the inference process
    - Best for: researchers, students learning how autoregressive generation works internally, debugging model behavior, custom decoding strategies
    - Our simple loop is 🐢 slow because it re-processes the whole sequence at every step; §7h shows how the **KV cache** fixes this

# <span style='background :lightgreen' >5. Inference using Transformers Library <b>pipeline()</b> Method</span>
<div style="text-align:center;">
    <img src="../images/pipeline-abstract.png"
         style="max-width:1500px; width:100%; height:auto; display:inline-block;">
</div>

- The pipeline below receives the chat messages directly, applies TinyLlama's chat template, generates the reply and decodes it, all in one call.
>- You may see some warnings from the transformers library while it runs (for example, that both `max_new_tokens` and `max_length` are set: TinyLlama's `generation_config.json` sets `max_length = 2048`, and `max_new_tokens` takes precedence). They are harmless.

In [10]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import pipeline


tokenizer = AutoTokenizer.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0")
model = AutoModelForCausalLM.from_pretrained(
                                            "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
                                            device_map="cpu",
                                            dtype="auto",
                                        )
# Create a pipeline for text generation
pipe = pipeline(
                task='text-generation',
                model=model,
                tokenizer=tokenizer,
                return_full_text=True, # True returns the whole conversation including the new reply; False returns only the reply
                )

# The pipeline accepts the list of chat messages directly and applies the model's chat template itself
messages = [{"role": "system", "content": "You are a helpful assistant."}, {"role": "user", "content": "Who is Ada Lovelace?"}]
generated = pipe(messages, max_new_tokens=100)          # max_new_tokens limits the length of the reply
print(generated[0]['generated_text'][-1]['content'])   # the last message in the returned conversation is the assistant's reply

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer LlamaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Ada Lovelace, also known as Lady Byron, was one of the pioneers of computer science and computer programming. A Lady Byron was a title that was bestowed on the children of Lord Byron (the poet, playwright, and politician) and was bestowed upon Ada and her brother Charles Babbage. Ada Lovelace was the daughter of Lord Byron and his first wife, Mary, and the niece of Lord Byron. She was born on


# <span style='background :lightgreen' >6. Inference using <b>generate()</b> Method</span>
- The `generate()` method exists for all models capable of token generation, which typically means causal / decoder-only or encoder-decoder models, but not encoder-only models. 
- The `generate()` autoregressively produces tokens by predicting one token at a time based on the input sequence (input_ids) and previously generated tokens.
- It supports sampling strategies (do_sample, temperature, top_k, top_p) to control randomness and diversity in generated text.
- You can control sequence length and stopping conditions using max_new_tokens and eos_token_id, and prevent repeated phrases with repetition_penalty.
- It also respects attention and padding via attention_mask and pad_token_id to ensure proper handling of input sequences.
```python
output_ids = model.generate(
    input_ids,                         # REQUIRED: torch.Tensor - token IDs of the input prompt
    do_sample=False,                   # OPTIONAL: bool - whether to use sampling; if False, uses greedy decoding (default: False)
    temperature=1.0,                   # OPTIONAL: float - softmax temperature for sampling; higher = more random (default: 1.0)
    top_k=50,                          # OPTIONAL: int - consider only top-k tokens at each step for sampling (default: 50)
    top_p=1.0,                         # OPTIONAL: float - nucleus sampling probability; consider tokens covering top_p probability mass (default: 1.0)
    attention_mask=None,               # OPTIONAL: torch.Tensor - 1 for real tokens, 0 for padding (default: None)
    max_new_tokens=None,               # OPTIONAL: int - maximum number of new tokens to generate (if not set, the limit in the model's generation_config.json is used, e.g., TinyLlama: max_length=2048 tokens incl. the prompt; if the model sets no limit, generation stops after 20 new tokens)
    eos_token_id=None,                 # OPTIONAL: int or List[int] - stop generation when this token is produced (default: model config eos_token_id)
    pad_token_id=None,                 # OPTIONAL: int - padding token ID, used for attention mask or batching (default: model config pad_token_id)
    num_return_sequences=1,            # OPTIONAL: int - number of generated sequences to return (default: 1)
    # ... 
)
```
- `temperature`, `top_k` and `top_p` are used only when `do_sample=True`. With greedy decoding (`do_sample=False`) they are ignored.
- `generate()` uses the **KV cache** by default (`use_cache=True`); §7h shows what that means and how much time it saves.

In [11]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

tokenizer = AutoTokenizer.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0")
model = AutoModelForCausalLM.from_pretrained(
                                            "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
                                            device_map="cpu",
                                            dtype="auto",
                                        )
messages = [{"role": "system", "content": "You are a helpful assistant."}, {"role": "user", "content": "Who is Ada Lovelace?"}]

# return_dict=True returns BOTH input_ids and attention_mask, ready to be passed to generate()
inputs = tokenizer.apply_chat_template(
                                        messages,
                                        tokenize=True,
                                        return_tensors="pt",
                                        return_dict=True,
                                        add_generation_prompt=True
                                        )

torch.manual_seed(42)                         # makes the random sampling below reproducible

# The generate() method autoregressively produces tokens based on input_ids to generate the response
output_ids = model.generate(
                            **inputs,                 # input_ids and attention_mask
                            max_new_tokens=100,       # Maximum number of tokens to generate (controls response length)
                            do_sample=True,           # Enable stochastic sampling (vs greedy generation)
                            temperature=0.7,          # Softmax temperature; higher = more random output
                            top_k=50,                 # Top-K sampling; only consider the top 50 tokens at each step
                            top_p=0.9,                # Nucleus sampling; consider tokens covering 90% probability mass
                            repetition_penalty=1.1,   # Penalize repeated tokens to avoid loops
                            eos_token_id=tokenizer.eos_token_id,  # Stop generation when this token is produced
                            pad_token_id=tokenizer.eos_token_id,  # Ensure proper padding for shorter sequences
                        )

# output_ids contains the prompt followed by the reply, so we skip the prompt tokens and decode only the reply
prompt_length = inputs["input_ids"].shape[1]
response = tokenizer.decode(output_ids[0, prompt_length:], skip_special_tokens=True)
print("Model reply:\n", response)

[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Model reply:
 Ada Lovelace was an English mathematician, writer, and inventor who made significant contributions to the development of computer science. She was the daughter of Charles Babbage, the inventor of the first mechanical general-purpose computer, the Analytical Engine. Despite facing gendered and societal barriers, she continued her father's work on algorithms and wrote some of the earliest code for a computer. Her contributions include:

1. Developing the first algorithm


>- **Check the facts!** TinyLlama is a small (1.1B-parameter) model and often *hallucinates*. In our run, the replies of both the pipeline (§5) and `generate()` (above) contain factual errors: they call Ada Lovelace the sister or the daughter of Charles Babbage. In fact, Ada Lovelace was the daughter of the poet Lord Byron, and she worked with Charles Babbage on his Analytical Engine. Larger models make such mistakes less often, but you should always verify important facts in an LLM's output.

# <span style='background :lightgreen' >7. Inference using a <b>Manual Forward Pass</b></span>

## a. Load the Tokenizer and Model Instance

In [12]:
from transformers import AutoTokenizer, AutoModelForCausalLM


tokenizer = AutoTokenizer.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0")
model = AutoModelForCausalLM.from_pretrained(
                                            "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
                                            device_map="cpu",
                                            dtype="auto",
                                        )

## b. Tokenize the text by converting raw text to token IDs that the model can understand and process

In [13]:
messages = [{"role": "system", "content": "You are a helpful assistant."}, {"role": "user", "content": "What is the capital of Pakistan?"}]
input_token_ids = tokenizer.apply_chat_template(
                                        messages,
                                        tokenize=True,             
                                        return_tensors="pt",       
                                        return_dict=False,          # transformers v5: without this it returns a BatchEncoding, not a tensor
                                        add_generation_prompt=True 
                                        )
input_token_ids

tensor([[  529, 29989,  5205, 29989, 29958,    13,  3492,   526,   263,  8444,
         20255, 29889,     2,    13, 29966, 29989,  1792, 29989, 29958,    13,
          5618,   338,   278,  7483,   310, 21215, 29973,     2,    13, 29966,
         29989,   465, 22137, 29989, 29958,    13]])

## c. Feed Token IDs to the model to Generate `logits`

In [14]:
import torch 

# Perform a forward pass through the language model
# torch.no_grad() tells PyTorch not to record operations for computing gradients (needed only for training): this saves memory and time
with torch.no_grad():
    outputs = model(
                    input_ids = input_token_ids,  # tensor containing token IDs of the prompt, these tokens are converted internally into embeddings and then passed through the transformer layers
                    attention_mask = torch.ones_like(input_token_ids) # attention_mask tells the model which tokens are REAL tokens and which are padding tokens.
                                                                        # 1 → token is valid and should be attended to
                                                                        # 0 → token is padding and should be ignored
                                                                        # since our prompt has no padding tokens, all positions are 1
                    )
# outputs is a structured object containing several elements such as logits, hidden_states, attentions (if requested)
# logits represent the model's raw prediction scores for the next token having a shape of (batch_size, sequence_length, vocabulary_size)
# for every token position in the input sequence, the model predicts a probability distribution over the ENTIRE vocabulary which are raw scores (not probabilities yet)
logits = outputs.logits
 
# prints the dimensions of the logits tensor: (1, seq_len, vocab_size), 
        # batch_size = 1 (one prompt), 
        # seq_len = number of tokens in the input prompt (36 here),
        # vocab_size = 32000  size of tokenizer vocabulary (~32k for TinyLlama)
print("Logits shape:", outputs.logits.shape)

print(logits)       # Each vector logits[0, i, :] contains the model's prediction scores for the NEXT token after position i in the sequence                    

Logits shape: torch.Size([1, 36, 32000])
tensor([[[-9.0000e+00, -8.8125e+00,  2.8906e+00,  ..., -3.5312e+00,
          -5.6562e+00, -5.5938e+00],
         [-1.5547e+00, -1.0703e+00,  2.4844e+00,  ..., -7.1094e-01,
          -2.2344e+00,  5.3516e-01],
         [-2.9375e+00, -2.0938e+00,  5.6562e+00,  ..., -1.5547e+00,
          -5.9688e+00, -3.1094e+00],
         ...,
         [-6.0000e+00, -4.8750e+00,  2.3906e+00,  ..., -5.2500e+00,
          -5.4688e+00, -1.2146e-02],
         [-6.7500e+00, -5.9688e+00,  1.2688e+01,  ..., -1.5781e+00,
          -6.4062e+00, -6.7969e-01],
         [-6.9062e+00, -7.5000e+00,  3.1875e+00,  ...,  6.7444e-03,
          -4.2500e+00, -9.6875e-01]]], dtype=torch.bfloat16)


In [15]:
# during generation, we typically take the LAST position: logits[0, -1, :] 
# then apply softmax to convert logits into probabilities and choose the next token (argmax / sampling / top-k / top-p)
last_token_logits = logits[:, -1, :]
print("Last token logits shape:", last_token_logits.shape)
print(last_token_logits)

Last token logits shape: torch.Size([1, 32000])
tensor([[-6.9062e+00, -7.5000e+00,  3.1875e+00,  ...,  6.7444e-03,
         -4.2500e+00, -9.6875e-01]], dtype=torch.bfloat16)


## d. Convert `logits` to Probabilities
- The a/m logits are just numbers — they can be large, small, positive, or negative. They do not represent probabilities yet.
- To turn logits into actual probabilities that sum to 1, we apply the softmax function.
- Before softmax (logits): `[-2.3, 5.1, -0.8, 3.2, ...]`  (can be negative, any range)
- After softmax (probs):   `[0.001, 0.82, 0.02, 0.12, ...]` (all positive, sum to 1.0)
- To summarize: **Softmax turns the model’s raw guesses (logits) into real probabilities that tell us which word the model thinks should come next.**

In [16]:
import torch.nn.functional as F # Import PyTorch's functional API module containing common neural network functions such as: softmax, relu, cross_entropy, etc.

# logits shape is (batch_size, sequence_length, vocabulary_size) and dim=-1 means apply softmax across the LAST dimension, which corresponds to the vocabulary dimension.
probs = F.softmax(logits, dim=-1) 

print("Probabilities shape:", probs.shape) # Print the shape of the probabilities tensor, which remains the same as logits: (batch_size, sequence_length, vocabulary_size)
                                                
print("\n", probs) # Print the full probability tensor, and  each row sums to 1.0 because of the softmax operation.

print("\nLast token probabilities are:\n", probs[0, -1 :]) # Extract and print the probability distribution for the LAST token in the sequence.
                                                                # 0  → first (and only) item in the batch
                                                                # -1 → last token position in the input sequence
                                                                # This vector contains the probabilities of every token in the vocabulary being the NEXT generated token.

Probabilities shape: torch.Size([1, 36, 32000])

 tensor([[[4.9768e-09, 6.0245e-09, 7.2861e-04,  ..., 1.1846e-06,
          1.4156e-07, 1.5087e-07],
         [1.8741e-13, 3.0376e-13, 1.0630e-11,  ..., 4.3521e-13,
          9.5035e-14, 1.5135e-12],
         [6.0396e-14, 1.4033e-13, 3.2560e-10,  ..., 2.3981e-13,
          2.9005e-15, 5.0626e-14],
         ...,
         [4.4187e-14, 1.3589e-13, 1.9463e-10,  ..., 9.3703e-14,
          7.5051e-14, 1.7621e-11],
         [6.7724e-15, 1.4766e-14, 1.8775e-06,  ..., 1.1937e-12,
          9.5479e-15, 2.9274e-12],
         [1.4850e-12, 8.2068e-13, 3.5856e-08,  ..., 1.4916e-09,
          2.1146e-11, 5.6389e-10]]], dtype=torch.bfloat16)

Last token probabilities are:
 tensor([[1.4850e-12, 8.2068e-13, 3.5856e-08,  ..., 1.4916e-09, 2.1146e-11,
         5.6389e-10]], dtype=torch.bfloat16)


## e. Picking the Next Two Tokens in the Output using Greedy Decoding
- The simplest way to generate next token is to use Greedy decoding, i.e., "Always pick the word with the highest probability. No randomness."
- The `torch.argmax()` method finds the index of the highest probability
- The `probs[0, -1, :]` extracts probabilities for the last token position
- The `unsqueeze(0)` just reshapes it

In [17]:
# torch.argmax() returns the index of the highest probability value, that corresponds to the token ID that the model predicts as the  most likely next token (greedy decoding).
# .unsqueeze(0) adds a new dimension at position 0. This converts the scalar into a 1-D tensor with shape (1,) which matches the format expected by the tokenizer.decode() function or when appending tokens during generation.
first_token_id = torch.argmax(probs[0, -1, :]).unsqueeze(0)

# Prints the numerical token ID from the model vocabulary. .item() extracts the raw Python integer from the tensor.
print("First token ID:", first_token_id.item())


# tokenizer.decode(...) converts token IDs back into human-readable text.
print("First token text:", tokenizer.decode(first_token_id))

First token ID: 1576
First token text: The


In [18]:
# Append the first predicted token to the input
new_input_ids = torch.cat([input_token_ids, first_token_id.unsqueeze(0)], dim=1)

# Forward pass again to get logits for the new (extended) sequence
with torch.no_grad():
    outputs2 = model(
            input_ids=new_input_ids,
            attention_mask=torch.ones_like(new_input_ids)
        )

# Get probabilities for the new last position (this will give the second word)
logits2 = outputs2.logits
probs2 = F.softmax(logits2, dim=-1)

# Pick the next most probable word → the second predicted token
second_token_id = torch.argmax(probs2[0, -1, :]).unsqueeze(0)

# Print token ID and decoded word
print("Second token ID:", second_token_id.item())
print("Second token text:", tokenizer.decode(second_token_id))

Second token ID: 7483
Second token text: capital


## f. Picking up to 50 Next Tokens in the Output using Greedy Decoding (Autoregressive Generation)
- Autoregressive means each new token depends on all previous tokens.
- We build the sequence one token at a time, feeding output back as input
    - Start with the original input token IDs and store them in generated_ids, which will gradually grow as the model generates new tokens.
    - In each iteration, the current sequence is fed into the model, and it returns logits for every position in the sequence (up to the last newly added token).
    - From the output logits, we take only the logits of the last position, because the model predicts the next token based on everything generated so far.
    - We apply softmax → probabilities → argmax to pick the most likely next token (greedy decoding).
    - The selected token is appended to generated_ids, increasing the sequence length by 1, and the loop repeats, generating one token at a time.
    - The loop **stops** as soon as the model produces the end-of-sequence token `</s>`. Without this check, a chat model keeps going after its answer and starts inventing the next turn of the conversation (e.g., a new question from the "user").
    - We also measure the time, so that we can compare it with the KV cache version in §7h.





In [19]:
generated_ids = input_token_ids.clone()  # Start with original input tokens
generated_ids

tensor([[  529, 29989,  5205, 29989, 29958,    13,  3492,   526,   263,  8444,
         20255, 29889,     2,    13, 29966, 29989,  1792, 29989, 29958,    13,
          5618,   338,   278,  7483,   310, 21215, 29973,     2,    13, 29966,
         29989,   465, 22137, 29989, 29958,    13]])

In [20]:
import time

# In each iteration of the following loop, one next token is added to the generated_ids array
start = time.perf_counter()
with torch.no_grad():                                                                    # no gradients are needed for inference
    for _ in range(50):
        outputs = model(
                        input_ids=generated_ids,  # This grows by 1 token each iteration
                        attention_mask=torch.ones_like(generated_ids)
                        )
        next_logits = outputs.logits                                                     # Logits for ALL positions, shape [1, current_seq_len, vocab_size]
        next_token = torch.argmax(F.softmax(next_logits[0, -1, :], dim=-1)).unsqueeze(0) # Take the LAST position, apply softmax and select the most probable token (greedy decoding).
                                                                                         # (softmax does not change which value is the largest, so argmax of the logits gives the same token)
        generated_ids = torch.cat([generated_ids, next_token.unsqueeze(0)], dim=1)       # Append the new token to the sequence, so the generated_ids grows: [1, seq_len] → [1, seq_len+1]
        if next_token.item() == tokenizer.eos_token_id:                                  # Stop as soon as the model produces the end-of-sequence token </s>
            break
time_no_cache = time.perf_counter() - start

print(f"Generated {generated_ids.shape[1] - input_token_ids.shape[1]} tokens in {time_no_cache:.2f} s (without KV cache)")
generated_ids

Generated 9 tokens in 3.85 s (without KV cache)


tensor([[  529, 29989,  5205, 29989, 29958,    13,  3492,   526,   263,  8444,
         20255, 29889,     2,    13, 29966, 29989,  1792, 29989, 29958,    13,
          5618,   338,   278,  7483,   310, 21215, 29973,     2,    13, 29966,
         29989,   465, 22137, 29989, 29958,    13,  1576,  7483,   310, 21215,
           338, 16427, 20143, 29889,     2]])

## g. Decode generated tokens back to human-readable text

In [21]:
response = tokenizer.decode(
                            generated_ids[0, input_token_ids.shape[1]:],   # Skip the prompt tokens: decode only the newly generated tokens
                            skip_special_tokens=True                       # Remove tokens like <s>, </s>, <pad>
                            )
print("\nModel reply:\n", response)


Model reply:
 The capital of Pakistan is Islamabad.


<br><br><br><br>

## h. Speeding Up Generation with the KV Cache
- Look again at the loop in part f: in every iteration we pass the **whole** sequence (the prompt plus all the tokens generated so far) through all 22 layers of TinyLlama. The keys and values of the earlier tokens are computed again and again, although they never change. This is why the manual forward pass is marked 🐢 slow in the table of §4.
- In Lec-20 §10 we learned the fix: the **KV cache**. When we call the model with `use_cache=True`, it also returns `past_key_values`: the keys and values of every token, in every layer. In the next step we pass this cache back to the model and feed it **only the new token**.
    - **Prefill:** run the whole prompt through the model once; this fills the cache.
    - **Decode:** in each step, feed only the newest token together with the cache; the model appends that token's keys and values to the cache.
- The code below repeats the greedy loop of part f with the KV cache, checks that it produces **exactly the same tokens**, and compares the time.

In [22]:
generated_kv = input_token_ids.clone()                      # start again from the prompt
start = time.perf_counter()
with torch.no_grad():
    # ── Prefill: process the whole prompt once and fill the cache ──
    outputs = model(input_ids=generated_kv, use_cache=True)
    past_key_values = outputs.past_key_values               # keys and values of all prompt tokens, for every layer

    for _ in range(50):
        next_token = torch.argmax(outputs.logits[0, -1, :]).reshape(1, 1)    # greedy decoding, shape [1, 1]
        generated_kv = torch.cat([generated_kv, next_token], dim=1)
        if next_token.item() == tokenizer.eos_token_id:                      # stop at the end-of-sequence token
            break
        # ── Decode: feed ONLY the new token, together with the cache ──
        outputs = model(input_ids=next_token, past_key_values=past_key_values, use_cache=True)
        past_key_values = outputs.past_key_values            # the cache now also holds the new token's keys and values
time_kv = time.perf_counter() - start

print(f"Generated {generated_kv.shape[1] - input_token_ids.shape[1]} tokens in {time_kv:.2f} s (with KV cache)")
print(f"Without the KV cache (part f) it took {time_no_cache:.2f} s  →  {time_no_cache / time_kv:.1f}x faster")
print("Exactly the same tokens as in part f?", torch.equal(generated_kv, generated_ids))
print("\nModel reply:\n", tokenizer.decode(generated_kv[0, input_token_ids.shape[1]:], skip_special_tokens=True))

Generated 9 tokens in 0.75 s (with KV cache)
Without the KV cache (part f) it took 3.85 s  →  5.1x faster
Exactly the same tokens as in part f? True

Model reply:
 The capital of Pakistan is Islamabad.


- The KV cache is an **exact** optimization: the generated tokens are identical, only the time changes.
- Next, let us look inside the cache. It has one entry per layer, and each entry holds a key tensor and a value tensor of shape (batch, KV heads, tokens, head_dim).

In [23]:
cache  = past_key_values
keys_0 = cache.layers[0].keys                                 # the keys stored for layer 0
config = model.config

print("Layers in the cache      :", len(cache.layers))
print("Keys of layer 0          :", tuple(keys_0.shape), "→ (batch, KV heads, tokens, head_dim)")
print("Query heads / KV heads   :", config.num_attention_heads, "/", config.num_key_value_heads)

# KV cache size = 2 (K and V) × layers × KV heads × head_dim × bytes per number   (per token)
bytes_per_token = 2 * config.num_hidden_layers * config.num_key_value_heads * config.head_dim * keys_0.element_size()
print(f"KV cache per token       : {bytes_per_token:,} bytes = {bytes_per_token / 1024:.0f} KiB")
print(f"KV cache at 2,048 tokens : {bytes_per_token * 2048 / 2**20:.0f} MiB  (TinyLlama's full context)")

Layers in the cache      : 22
Keys of layer 0          : (1, 4, 44, 64) → (batch, KV heads, tokens, head_dim)
Query heads / KV heads   : 32 / 4
KV cache per token       : 22,528 bytes = 22 KiB
KV cache at 2,048 tokens : 44 MiB  (TinyLlama's full context)


- TinyLlama has 32 query heads but only **4 KV heads**: it uses **Grouped-Query Attention** (Lec-20 §10g), and each K, V head is shared by 8 query heads. With standard multi-head attention (32 KV heads) the cache would be 8× larger: 176 KiB per token, i.e., 352 MiB at 2,048 tokens.
- The number of tokens in the cache is the prompt plus every generated token that was fed back to the model (the final `</s>` is never fed back).
- Finally, `generate()` uses the KV cache by default. Let us switch it off and on and compare. To make the comparison fair, both runs generate exactly 100 new tokens (`min_new_tokens=100`).

In [24]:
def timed_generate(use_cache):
    start = time.perf_counter()
    output_ids = model.generate(
                                input_ids=input_token_ids,
                                attention_mask=torch.ones_like(input_token_ids),
                                max_new_tokens=100,
                                min_new_tokens=100,          # force exactly 100 new tokens in both runs
                                do_sample=False,
                                use_cache=use_cache,
                                pad_token_id=tokenizer.eos_token_id,
                                )
    return time.perf_counter() - start, output_ids

timed_generate(True)                                         # warm-up run, so that both timings are fair
for use_cache in (True, False):
    seconds, output_ids = timed_generate(use_cache)
    new_tokens = output_ids.shape[1] - input_token_ids.shape[1]
    print(f"use_cache={str(use_cache):<5} → {seconds:6.2f} s for {new_tokens} new tokens")

[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


use_cache=True  →   4.56 s for 100 new tokens


use_cache=False →  88.23 s for 100 new tokens


- Without the cache, every step re-processes the prompt and all earlier tokens, so each step is slower than the previous one. With the cache, each step processes only one new token. The longer the generated text, the larger the speed-up.
- The price is memory: the cache grows by one entry per token in every layer. For large models with long contexts the cache can become as large as the model itself (Lec-20 §10f).

## Memory Cleanup (Optional but recommended)

In [25]:
#import gc, torch
#del model, input_token_ids, outputs, logits, probs, generated_ids, generated_kv, past_key_values
#gc.collect()                                                # Force Python garbage collection
#torch.cuda.empty_cache()                                    # Clear PyTorch's CUDA cache (if using an NVIDIA GPU)
#torch.mps.empty_cache()                                     # Clear PyTorch's MPS cache (if using an Apple Silicon GPU)